# Laboratorio: formas cuadráticas y positividad

Construiremos matrices simétricas desde polinomios, clasificaremos formas por su espectro y aplicaremos congruencias, factorizaciones y optimización cuadrática.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=6, suppress=True)

## 1. Matriz simétrica desde una expresión

Para una forma cuadrática, la mitad de la Hessiana es su matriz simétrica asociada. Esto reparte automáticamente los coeficientes cruzados.

In [ ]:
def matriz_forma(Q, variables):
    Q = sp.expand(Q)
    A = sp.simplify(sp.hessian(Q, variables) / 2)
    x = sp.Matrix(variables)
    if sp.expand((x.T*A*x)[0] - Q) != 0:
        raise ValueError("La expresión no es una forma cuadrática homogénea.")
    return A

x, y = sp.symbols('x y', real=True)
Q = 5*x**2 + 4*x*y + y**2
A = matriz_forma(Q, (x, y))
print("Matriz simétrica asociada:")
sp.pprint(A)
assert A == sp.Matrix([[5, 2], [2, 1]])

## 2. Clasificación espectral

La función siguiente clasifica una matriz simétrica a partir de los signos de sus valores propios exactos.

In [ ]:
def clasificar_simetrica(A):
    A = sp.Matrix(A)
    if A != A.T:
        raise ValueError("Se requiere una matriz simétrica.")
    valores = []
    for lam, mult in A.eigenvals().items():
        valores.extend([lam] * mult)
    positivos = sum(bool(lam.is_positive) for lam in valores)
    negativos = sum(bool(lam.is_negative) for lam in valores)
    ceros = sum(lam == 0 for lam in valores)
    n = len(valores)
    if positivos == n:
        clase = 'definida positiva'
    elif negativos == n:
        clase = 'definida negativa'
    elif negativos == 0 and ceros > 0:
        clase = 'semidefinida positiva'
    elif positivos == 0 and ceros > 0:
        clase = 'semidefinida negativa'
    elif positivos > 0 and negativos > 0:
        clase = 'indefinida'
    else:
        clase = 'no decidida simbólicamente'
    return valores, (positivos, negativos, ceros), clase

A_ind = sp.Matrix([[1, sp.Rational(-3,2)], [sp.Rational(-3,2), 1]])
valores, inercia, clase = clasificar_simetrica(A_ind)
print("Valores propios:", valores)
print("Inercia:", inercia, "Clasificación:", clase)
assert set(valores) == {sp.Rational(-1,2), sp.Rational(5,2)}
assert inercia == (1, 1, 0) and clase == 'indefinida'

## 3. Reducción ortogonal

Formamos una matriz ortogonal con vectores propios normalizados y verificamos que el cambio $z=P^Tx$ elimina los términos cruzados.

In [ ]:
datos = A_ind.eigenvects()
columnas, diagonal = [], []
for lam, _, base in datos:
    q = sp.simplify(base[0] / sp.sqrt((base[0].T*base[0])[0]))
    columnas.append(q)
    diagonal.append(lam)
P = sp.Matrix.hstack(*columnas)
D = sp.diag(*diagonal)
assert sp.simplify(P.T*P) == sp.eye(2)
assert sp.simplify(P.T*A_ind*P) == D
print("P =")
sp.pprint(P)
print("D =")
sp.pprint(D)

## 4. Una familia con parámetro

Para $Q=x^2+y^2+z^2+2axz$, la clasificación cambia cuando alguno de los valores propios $1+a$ y $1-a$ atraviesa cero.

In [ ]:
a = sp.symbols('a', real=True)
A_a = sp.Matrix([[1, 0, a], [0, 1, 0], [a, 0, 1]])
print("Polinomio característico:")
sp.pprint(sp.factor(A_a.charpoly().as_expr()))
print("Valores propios:", A_a.eigenvals())
assert set(A_a.eigenvals()) == {1, 1-a, 1+a}

for valor, esperado in [(0, 'definida positiva'),
                        (1, 'semidefinida positiva'),
                        (-1, 'semidefinida positiva'),
                        (2, 'indefinida')]:
    _, _, resultado = clasificar_simetrica(A_a.subs(a, valor))
    assert resultado == esperado

## 5. Criterio de Sylvester

Calculamos los menores principales líderes. También comprobamos por qué reemplazar estrictamente positivo por no negativo no sirve para caracterizar semidefinitud.

In [ ]:
def menores_lideres(A):
    A = sp.Matrix(A)
    return [sp.factor(A[:k, :k].det()) for k in range(1, A.rows+1)]

A_pos = sp.Matrix([[2, -1], [-1, 2]])
print("Menores líderes de A_pos:", menores_lideres(A_pos))
assert menores_lideres(A_pos) == [2, 3]

contra = sp.diag(0, -1)
print("Menores líderes del contraejemplo:", menores_lideres(contra))
print("Clasificación:", clasificar_simetrica(contra)[2])
assert menores_lideres(contra) == [0, 0]
assert clasificar_simetrica(contra)[2] == 'semidefinida negativa'

## 6. Congruencia e inercia

Una congruencia invertible puede cambiar los valores propios, pero debe conservar cuántos son positivos, negativos y cero.

In [ ]:
A0 = sp.diag(1, -2, 0)
C = sp.Matrix([[1, 1, 0], [0, 1, 1], [1, 0, 1]])
assert C.det() != 0
B0 = sp.simplify(C.T*A0*C)

def inercia_numerica(M, tol=1e-10):
    vals = np.linalg.eigvalsh(np.array(M, dtype=float))
    return (int(np.sum(vals > tol)), int(np.sum(vals < -tol)),
            int(np.sum(np.abs(vals) <= tol)))

print("Valores propios de A0:", np.linalg.eigvalsh(np.array(A0, dtype=float)))
print("Valores propios de C.T A0 C:", np.linalg.eigvalsh(np.array(B0, dtype=float)))
assert inercia_numerica(A0) == inercia_numerica(B0) == (1, 1, 1)

## 7. Matrices de Gram y factorización semidefinida

Toda matriz $X^TX$ es semidefinida positiva. Recíprocamente, una matriz semidefinida positiva admite una factorización espectral $BB^T$.

In [ ]:
X = sp.Matrix([[1, 2, 0], [0, 1, 1]])
G = X.T*X
print("G = X.T X:")
sp.pprint(G)
assert G == G.T
assert all(lam >= 0 for lam in G.eigenvals())
assert G.rank() == X.rank()

A_psd = sp.Matrix([[1, 1], [1, 1]])
Qp, Dp = A_psd.diagonalize(normalize=True)
B_psd = sp.simplify(Qp * sp.diag(*[sp.sqrt(lam) for lam in Dp.diagonal()]))
assert sp.simplify(B_psd*B_psd.T - A_psd) == sp.zeros(2)

## 8. Optimización cuadrática

Minimizamos $f(x)=\frac12x^TAx-b^Tx$ con una matriz definida positiva y verificamos la identidad que certifica el mínimo global.

In [ ]:
A_opt = sp.Matrix([[2, -1], [-1, 2]])
b1, b2 = sp.symbols('b_1 b_2', real=True)
b = sp.Matrix([b1, b2])
x_estrella = sp.simplify(A_opt.inv()*b)
f_min = sp.simplify(-sp.Rational(1,2)*(b.T*A_opt.inv()*b)[0])
print("Punto minimizador:")
sp.pprint(x_estrella)
print("Valor mínimo:")
sp.pprint(f_min)

u, v = sp.symbols('u v', real=True)
h = sp.Matrix([u, v])
xx = x_estrella + h
f = lambda z: sp.Rational(1,2)*(z.T*A_opt*z)[0] - (b.T*z)[0]
assert sp.simplify(f(xx) - f(x_estrella) - sp.Rational(1,2)*(h.T*A_opt*h)[0]) == 0

## 9. Geometría de la clasificación

Comparamos una forma definida positiva, una semidefinida positiva y una indefinida mediante sus curvas de nivel.

In [ ]:
grid = np.linspace(-2.5, 2.5, 401)
XX, YY = np.meshgrid(grid, grid)
formas = [
    (XX**2 + 2*YY**2, 'definida positiva'),
    ((XX + YY)**2, 'semidefinida positiva'),
    (XX**2 - YY**2, 'indefinida'),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
niveles = [-6, -3, -1, 0, 1, 3, 6]
for ax, (ZZ, titulo) in zip(axes, formas):
    cont = ax.contour(XX, YY, ZZ, levels=niveles, cmap='coolwarm')
    ax.clabel(cont, fontsize=8)
    ax.axhline(0, color='black', lw=0.5)
    ax.axvline(0, color='black', lw=0.5)
    ax.set(title=titulo, xlabel='$x$', ylabel='$y$', aspect='equal')
plt.tight_layout()
plt.show()

## 10. Actividades

1. Implementa una prueba de definición negativa mediante los signos alternados de los menores líderes.
2. Clasifica las cuatro formas cuadráticas propuestas en la lista PC2 2026-I y encuentra vectores que exhiban cada signo.
3. Comprueba que la suma de dos matrices definidas positivas generadas aleatoriamente sigue siendo definida positiva.
4. Construye una matriz singular $C$ y verifica que $C^TAC$ pierde definición estricta aunque $A$ sea definida positiva.
5. Compara la raíz espectral de una matriz definida positiva con su factor de Cholesky.

## 11. Cierre

- Toda forma cuadrática real tiene una matriz simétrica asociada única.
- El teorema espectral la convierte en suma y diferencia de cuadrados.
- Los signos de los valores propios determinan la clasificación.
- Las congruencias preservan la inercia, no los valores propios.
- La definición positiva conecta álgebra lineal, geometría y optimización convexa.